<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 11 — Finding Things

Nearly every program looks something up: is this word in the list, which
record has this id, have I seen this customer before. You have written `in`
and `records[key]` all term without asking what they cost. This session asks,
because the answer decides whether a program finishes in a second or is still
running at lunch.

Five ideas, one part each:

| Part | Idea | In one sentence |
|---|---|---|
| 1 | Count the work, do not time it | The number of comparisons is exact and shows how the work grows. |
| 2 | Linear search | Look at every item: the work grows with the list. |
| 3 | Binary search | Halve a sorted list: a million items in twenty steps. |
| 4 | Hashing | Compute where the key must be and look only there. |
| 5 | Choosing | Each structure answers some questions quickly and others slowly or not at all. |

**How this session's notebooks fit together**

| Notebook | What it is |
|---|---|
| `01_Finding_Things` (this one) | The ideas, on lists you can follow by hand |
| `02_Yelp_Tips_Example` | One real task done three ways: matching 900,000 tips to their businesses |
| `03_Search_and_Hashing_Reference` | A lookup guide to keep open while you do Homework 10 |

**What is not repeated here.** Lists, dictionaries and sets as you use them
are Sessions 3 and 4. Sorting was Homework 3. Joining tables was Sessions 7
and 8. This session is about what happens inside them.

**Before you start**

- Set the kernel (upper right corner) to **Conda Python 3.12**.
- Run the cells in order with **Shift + Enter**. Before each cell, guess what
  it will show.

In [ ]:
import bisect
import math
import time

import matplotlib.pyplot as plt

---

## Part 1 — Count the work, do not time it

Two programs give the same answer. Which is better? The obvious test is a
stopwatch.

In [ ]:
numbers = list(range(1_000_000))

for attempt in range(5):
    start = time.perf_counter()
    found = 999_999 in numbers
    print("attempt {}: {:.4f} seconds".format(attempt + 1, time.perf_counter() - start))

Five runs of the same line, five slightly different times. On a cluster
shared with a hundred other people, on a busy afternoon, they differ a lot
more. A time depends on the machine, on what
else it is doing, and on luck.

There is a steadier measure: **count the steps**. For a search, the step that
matters is comparing an item with the target. A count is the same on every
machine and on every run, and it answers the question a time cannot: *what
happens when the list is ten times as long?*

In [ ]:
def linear_search(items, target):
    """Return (index, comparisons); index is -1 when target is not there."""
    comparisons = 0
    for index in range(len(items)):
        comparisons += 1
        if items[index] == target:
            return (index, comparisons)
    return (-1, comparisons)


names = ["Mia", "Ben", "Zoe", "Ana", "Tom", "Eli", "Kai", "Lea"]
print(linear_search(names, "Tom"))
print(linear_search(names, "Sam"))

Tom is the fifth name: five comparisons. Sam is not there, and it took all
eight to be sure. A function that returns its own count is the tool for the
rest of this notebook, and for Homework 10.

**✏️ Try it:** Which name costs the fewest comparisons, which the most, and what is the average over all eight names?

---

## Part 2 — Linear search: the work grows with the list

Search for the last item, the worst case, in lists of growing size.

In [ ]:
for size in (10, 100, 1_000, 10_000, 100_000):
    items = list(range(size))
    index, comparisons = linear_search(items, size - 1)
    print("{:>8,} items: {:>8,} comparisons".format(size, comparisons))

Ten times the list, ten times the work. That is what **linear** means: the
work is proportional to the size.

| | Comparisons, for a list of n |
|---|---|
| best case: the target is first | 1 |
| on average, when the target is there | about n / 2 |
| worst case: the target is last | n |
| **the target is not there** | **n, always** |

The last row matters most. "Is this new?" and "which of these are missing?"
are questions about things that are *not* there, and linear search pays full
price for every one.

`in`, `.index()` and `.count()` on a list are linear searches. So is this,
which looks innocent and is the usual reason a program that worked on a
thousand rows never finishes on a million:

```python
for order in orders:                      # 1,000,000 orders
    if order["customer"] in customers:    # a list of 100,000: a scan every time
        ...
```

A million scans of up to a hundred thousand items each: up to a hundred
billion comparisons.

**✏️ Try it:** How many comparisons does that loop make at most? At ten million comparisons a second, how long is that?

---

## Part 3 — Binary search: halve a sorted list

I am thinking of a number from 1 to 100. You may guess, and I will say
"higher" or "lower". Nobody guesses 1, 2, 3, ... People say 50. Then 75 or
25. Each answer cuts what is left in half.

That works because the numbers are **in order**, so one comparison tells you
about half of them at once. A sorted list allows the same thing.

In [ ]:
def binary_search(items, target, show=False):
    """Return (index, comparisons).  items must be sorted."""
    low, high = 0, len(items) - 1
    comparisons = 0
    while low <= high:
        middle = (low + high) // 2
        comparisons += 1
        if show:
            print("  round {}: looking between {} and {}, middle {} holds {!r}".format(
                comparisons, low, high, middle, items[middle]))
        if items[middle] == target:
            return (middle, comparisons)
        if items[middle] < target:
            low = middle + 1                 # the target is to the right
        else:
            high = middle - 1                # the target is to the left
    return (-1, comparisons)


in_order = sorted(names)
print(in_order)
print(binary_search(in_order, "Tom", show=True))

`low` and `high` fence in the part where the target can still be. Each round
looks at the middle of that part and moves one fence past it. The promise the
loop keeps: *if the target is in the list, it is between `low` and `high`*.
When the fences cross, nothing is left, and the target is not there.

In [ ]:
print(binary_search(in_order, "Sam", show=True))

In [ ]:
for size in (10, 100, 1_000, 10_000, 100_000, 1_000_000):
    items = list(range(size))
    index, comparisons = binary_search(items, size - 1)
    print("{:>10,} items: {:>3} comparisons   (log2 of the size is {:.1f})".format(
        size, comparisons, math.log2(size)))

Ten times the list, about three more comparisons. A million items take
twenty, because halving a million twenty times leaves one. The number of
halvings is the **logarithm to base 2** of the size, and work that grows this
way is called **logarithmic**.

The two growth laws, side by side:

In [ ]:
sizes = [10, 100, 1_000, 10_000, 100_000, 1_000_000]

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.plot(sizes, [size / 2 for size in sizes], marker="o", label="linear search, on average: n / 2")
ax.plot(sizes, [math.log2(size) for size in sizes], marker="s", label="binary search: log2(n)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("items in the list (logarithmic scale)")
ax.set_ylabel("comparisons per lookup (logarithmic scale)")
ax.set_title("The gap is not a constant factor. It widens.")
ax.legend()
fig.tight_layout()

Both axes are logarithmic, or the lower line would lie on the axis.

### The price: sorted data

Binary search on an unsorted list is not slow. It is **wrong**, silently:

In [ ]:
print(names)
print(binary_search(names, "Mia"))

Mia is the first item, and the search reports −1. It looked at the middle,
found a name that sorts before "Mia", and threw away the left half, where Mia
was. No error, no warning.

So binary search comes with a bill: the list has to be sorted first, and kept
sorted. Sorting n items takes about n × log2(n) comparisons. Whether that is
worth paying depends on how many lookups follow:

| Lookups in a list of 100,000 | Linear | Sort once, then binary |
|---:|---:|---:|
| 1 | 50,000 | 1,700,000 |
| 10 | 500,000 | 1,700,000 |
| 100 | 5,000,000 | 1,700,000 |
| 100,000 | 5,000,000,000 | 3,400,000 |

For one lookup, sorting is a waste. For a hundred it has already paid.

### Where would it go?

A small change to the loop answers a more useful question: at which position
does the target **belong**? Python has it built in, as `bisect`.

In [ ]:
scores = [10, 20, 20, 20, 30, 40]

for target in (20, 25, 99):
    print("{:2}: bisect_left {}   bisect_right {}".format(
        target, bisect.bisect_left(scores, target), bisect.bisect_right(scores, target)))

`bisect_left` gives the first position whose item is not below the target: the
first 20, or the place where 25 would be inserted. `bisect_right` gives the
position just after the last equal item. Two of them fence in a range without
looking inside it:

In [ ]:
low, high = 20, 40
first = bisect.bisect_left(scores, low)
past = bisect.bisect_left(scores, high)
print("scores from 20 up to (not including) 40:", past - first, "->", scores[first:past])

Order gives you more than "is it there". It gives you *between*, *before*,
*after*, *the nearest*. Remember that when Part 4 looks faster.

**✏️ Try it:** How many rounds does the number-guessing game need at most for 1 to 100? For 1 to 1,000,000,000? Use math.log2.

---

## Part 4 — Hashing: compute where it must be

Binary search rules things out quickly. The third idea is more radical: **do
not search at all.**

A cloakroom has 100 numbered hooks. A careless attendant hangs each coat on
any free hook and later walks along the rail looking for it: linear search. A
better one sorts the coats by owner's name: binary search, and a lot of
shuffling whenever a coat arrives. The best one uses a rule: *take the
ticket number, the last two digits are the hook.* Coat 4,217 goes on hook 17.
To fetch it, compute 17 and reach for it. No looking.

For that, a key must become a number. A **hash function** does it.

In [ ]:
def hash_key(key):
    """Turn a string into a whole number, the same one every time."""
    value = 0
    for character in key:
        value = (value * 31 + ord(character)) % 1_000_000_007
    return value


for name in names:
    print("{:4} hash {:>9,}   hook {} of 8".format(name, hash_key(name), hash_key(name) % 8))

`ord` gives a character's code; the loop mixes the codes so that the order of
the letters matters. `% 8` folds the big number onto 8 hooks.

Eight names, eight hooks, and already several names share a hook while other
hooks stay empty. With more possible
keys than hooks that is unavoidable. It is called a **collision**, and the
simplest remedy is to let a hook hold several coats: each slot becomes a
**bucket**, a short list.

In [ ]:
def build(keys, buckets):
    table = [[] for number in range(buckets)]            # separate lists!
    for position, key in enumerate(keys):
        table[hash_key(key) % buckets].append([key, position])
    return table


def table_get(table, key):
    """Return (value, comparisons)."""
    bucket = table[hash_key(key) % len(table)]           # go straight to it
    comparisons = 0
    for pair in bucket:
        comparisons += 1
        if pair[0] == key:
            return (pair[1], comparisons)
    return (None, comparisons)


table = build(names, 8)
for number, bucket in enumerate(table):
    print("bucket", number, bucket)

print()
print("Tom:", table_get(table, "Tom"))
print("Sam:", table_get(table, "Sam"))

To find Tom, compute his bucket and search **only that bucket**. To learn that
Sam is absent, look in one bucket and find it does not hold him. The other
seven are never touched.

Inside a bucket it is a linear search again. The method is fast as long as the
buckets are short, which depends on one number, the **load factor**: keys
divided by buckets.

In [ ]:
keys = ["key-{}".format(number) for number in range(10_000)]

print("{:>8} {:>12} {:>24}".format("buckets", "load factor", "comparisons per lookup"))
for buckets in (16, 256, 4_096, 16_384, 65_536):
    table = build(keys, buckets)
    total = sum(table_get(table, key)[1] for key in keys)
    print("{:>8,} {:>12.2f} {:>24.2f}".format(buckets, len(keys) / buckets, total / len(keys)))

With 16 buckets, a lookup wades through hundreds of keys: a slow linear search
with extra steps. With more buckets than keys it compares barely more than
one. A real hash table therefore **grows**: when the load factor passes a
limit, it doubles its buckets and puts every key in its new place.

Now the same growth table as before, with the third column:

| Items | Linear, average | Binary | Hash table |
|---:|---:|---:|---:|
| 1,000 | 500 | 10 | about 1 |
| 1,000,000 | 500,000 | 20 | about 1 |
| 1,000,000,000 | 500,000,000 | 30 | about 1 |

Work that does not grow with the size is called **constant**.

### You have been using one all along

Python's `dict` and `set` are hash tables. That is what makes them fast, and
it is the difference between these two lines:

In [ ]:
as_list = list(range(1_000_000))
as_set = set(as_list)

start = time.perf_counter()
for target in range(999_000, 999_100):
    found = target in as_list
list_seconds = time.perf_counter() - start

start = time.perf_counter()
for target in range(999_000, 999_100):
    found = target in as_set
set_seconds = time.perf_counter() - start

print("100 lookups in a list: {:.4f} s".format(list_seconds))
print("100 lookups in a set:  {:.6f} s".format(set_seconds))
print("the set is about {:,.0f} times faster".format(list_seconds / set_seconds))

Same `in`, same answer. The fix for the slow loop in Part 2 is one word:
`customers = set(customers)`.

Two consequences of how hashing works explain rules you have met without a
reason.

**A key must not change.** Its bucket is computed from its contents. If a list
could be a key and you appended to it afterwards, it would sit in the wrong
bucket and never be found. So Python allows only **immutable** things as keys:

In [ ]:
prices = {}
prices[("espresso", "small")] = 2.5          # a tuple is fine
try:
    prices[["espresso", "small"]] = 2.5      # a list is not
except TypeError as error:
    print("TypeError:", error)

**A hash table has no order to search by.** The hash scatters keys on purpose:
"Tom" and "Ton" land nowhere near each other. A table answers "is exactly this
key here?" and nothing else. It cannot tell you which keys start with "To",
which is the next larger, or how many lie between two values. Sorted data can.

One oddity: Python's own `hash("Tom")` gives a different number every time
Python starts, as a defence against attacks on web servers. It is consistent
within a run, which is all a `dict` needs. Homework 10 has you write your own
so that a table comes out the same every time.

**✏️ Try it:** Build the table of `names` with 2 buckets, then with 64. How long is the longest bucket each time?

---

## Part 5 — Choosing

| | Unsorted list | Sorted list | Hash table (`dict`, `set`) |
|---|---|---|---|
| Is this key here? | n | log n | 1 |
| Add a key | 1 (append) | n (make room) | 1 |
| Which keys are between a and b? | n | log n, then read them off | n: no order to use |
| The smallest, the next larger | n | 1, log n | n |
| All of them, in order | sort first | already | sort first |
| Needs | nothing | sorting; comparable items | hashable (immutable) keys; extra memory |

The numbers are how the work grows, with constants left out.

How to choose:

1. **Many lookups by exact key?** A `dict` or a `set`. This is the usual case,
   and the usual fix for a slow program.
2. **Questions about order or ranges?** Sort once, then `bisect`.
3. **A handful of items, or a single lookup?** A list and `in` are fine.
   Nothing here matters for ten items.

And before optimising anything: find the loop inside a loop. A linear search
is harmless once. Inside a loop over a million rows it is a million of them.

---

## Exercises

### 🧩 Exercise 1 — Count it

Write `count_matches(items, targets)` that returns how many of the targets
are in `items`, together with the total number of comparisons, using
`linear_search` from Part 1. Try it with 1,000 items and 1,000 targets of
which half are present. How many comparisons? Predict the number for 10,000
and 10,000 before you run it.

### 🧩 Exercise 2 — Follow a search

Take `sorted_numbers = list(range(0, 200, 3))`. By hand, on paper, follow
`binary_search` for the target 99: write down `low`, `high` and `middle` for
each round. Then run it with `show=True` and compare. Do the same for 100,
which is not in the list.

### 🧩 Exercise 3 — When does sorting pay?

A list has 50,000 items. A linear search costs 25,000 comparisons on average;
a binary search about 16; sorting about 50,000 × log2(50,000). After how many
lookups has sorting paid for itself? Write the inequality, then compute it.

### 🧩 Exercise 4 — A bad hash function

Replace `hash_key` with a version that returns `len(key)`, rebuild the table
of the 10,000 keys of Part 4 with 16,384 buckets, and measure the comparisons
per lookup. What went wrong, in one sentence? Then try a version that returns
the sum of the character codes.

---

## Summary

| Idea | The habit | The tool |
|---|---|---|
| Count the work | Ask how it grows, not how long it took today | a counter in the loop |
| Linear search | Find the loop inside a loop | `in`, `.index` on a list |
| Binary search | Sort once when many lookups follow; never on unsorted data | `bisect.bisect_left` |
| Hashing | For lookups by exact key, use a table | `dict`, `set` |
| Choosing | Name the question: exact key, or order? | the table in Part 5 |

### Where each idea appears in Homework 10

| Idea | Homework 10 |
|---|---|
| A search that returns its count | Phase 1, `linear_search` |
| Halving, and the promise the loop keeps | Phase 2, `binary_search` |
| Where it would go; a range from its two ends | Phase 3, `leftmost`, `count_in_range` |
| A key becomes a number | Phase 4, `hash_key`, `bucket_index` |
| Buckets and collisions | Phase 5, `new_table`, `table_put`, `table_get` |
| Load factor and growing | Phase 6, `load_factor`, `table_resize`, `build_table` |
| The growth table, and when sorting pays | Phase 7, `compare_methods`, `break_even` |

**Next:** `02_Yelp_Tips_Example` runs the three methods on 900,000 real
lookups, and `03_Search_and_Hashing_Reference` is the lookup guide.